In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [2]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True # 데이터 역직렬화 허용
)

In [3]:
retriever = vectorstore.as_retriever()

In [5]:
from langchain.tools import tool

@tool
def search_documents(query: str) -> str:
    """2026 년 테크노빌드 주식회사(TechnoBuild) 임직원 
통합 가이드북입니다.
    """
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [6]:
system_prompt = """당신은 테크노빌드 주식회사 가이드북 정보를 친절하게 제공하는 어시스턴트입니다.

1. 정보가 필요할 경우 반드시 검색 도구(retriever_tool)를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 억지로 꾸며내지 말고 모른다고 답변하세요.
"""

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_documents],
    system_prompt=system_prompt
)

In [8]:
query = "자격증 비용은 얼마를 받을 수 있어?"

In [10]:
from langchain.messages import SystemMessage, HumanMessage

res = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

In [17]:
print(res["messages"][-1].content[0]["text"])

테크노빌드 주식회사에서는 직무 관련 국가 기술 자격을 취득할 경우, **자격 등급에 따라 1회성 축하금과 매월 지급되는 자격 수당**을 지원하고 있습니다. 상세 내용은 다음과 같습니다.

### **자격증 등급별 지원 금액**

| 자격 등급 | 축하금 (1회성) | 자격 수당 (월) | 대상 자격증 예시 |
| :--- | :--- | :--- | :--- |
| **기술사 / 기능장** | **200만 원** | **30만 원** | 금속재료, 용접, 기계가공 등 |
| **기사** | **50만 원** | **10만 원** | 일반기계, 전기, 산업안전 등 |
| **산업기사** | **30만 원** | **5만 원** | 기계설계, 위험물 등 |
| **기능사** | **10만 원** | **3만 원** | 선반, 밀링, 특수용접 등 |

### **지급 조건 및 방법**
*   **자격 수당:** 동일 등급 내에서는 1개의 자격증만 수당으로 인정됩니다. (상위 등급 취득 시 수당이 갱신됩니다.)
*   **축하금:** 취득 횟수에 제한 없이 지급됩니다.
*   **신청 절차:** 자격증 사본을 HR팀에 제출해야 합니다.
    *   **축하금:** 제출 후 2주 이내에 별도로 입금됩니다.
    *   **자격 수당:** 제출한 다음 달 급여부터 반영되어 지급됩니다.
